# 01 — From the collected data to the final table

**What this notebook does.** It takes the intermediate tables already computed (folder `data/interim`) and builds **one final table**, `data/processed/cities.csv`: **one row per city**, with all the variables needed by the three blocks of the project (exploration, clustering, classification).

**Why we start from `data/interim` and not from the raw data.** The raw data (about 1 GB of Parquet files from the European Environment Agency) are not on GitHub. They were summarised per station and per year by the scripts in `src/` (see the README). Here we only work with small CSV files, which can also be opened in Excel.

**Rules already applied before this point** (decision log in the README):
- only **urban or suburban background** stations (not roadside ones);
- a station counts for a year if it has at least **75% of the days** (PM) or of the hours (NO₂, ozone);
- period **2016–2023**; we keep the cities with **at least 6 valid years out of 8 for each pollutant** (PM10, NO₂ and ozone, counted separately: the 6 years need not be the same for the three pollutants, see section 1).

**Structure:**

0. The sources at a glance
1. Cities and means
2. Pollution signature
3. Weather
4. Geography
5. Target for block C
6. Checks
7. Saving

## 0. The sources at a glance

The project joins four kinds of data. The unit of analysis is the **city**: everything is brought to one value per city.

<table style="margin-left:0; margin-right:auto">
<tr><th style="text-align:left">What</th><th style="text-align:left">Source</th><th style="text-align:left">Period</th><th style="text-align:left">Resolution</th><th style="text-align:left">How it is linked to a city</th></tr>
<tr><td style="text-align:left">PM10, PM2.5</td><td style="text-align:left">EEA, validated E1a data</td><td style="text-align:left">2016–2023</td><td style="text-align:left">daily (some hourly) values per sampling point</td><td style="text-align:left">the EEA assigns each sampling point to a city (Urban Audit); we keep urban and suburban <b>background</b> stations</td></tr>
<tr><td style="text-align:left">NO₂, ozone (Italy)</td><td style="text-align:left">EEA, validated E1a data</td><td style="text-align:left">2016–2023</td><td style="text-align:left">hourly values per sampling point</td><td style="text-align:left">same as above</td></tr>
<tr><td style="text-align:left">NO₂, ozone (other countries)</td><td style="text-align:left">EEA station data on Zenodo (record 17085064)</td><td style="text-align:left">2016–2023</td><td style="text-align:left">yearly indicators and daily values per station</td><td style="text-align:left">same as above (station metadata of the EEA)</td></tr>
<tr><td style="text-align:left">Weather</td><td style="text-align:left">NASA POWER (MERRA-2 reanalysis)</td><td style="text-align:left">2016–2023</td><td style="text-align:left">monthly means on cells of about 50 km</td><td style="text-align:left">value at the city's coordinates (mean position of its background stations)</td></tr>
<tr><td style="text-align:left">Distance from the sea</td><td style="text-align:left">Natural Earth coastline 1:50m</td><td style="text-align:left">—</td><td style="text-align:left">one value per city</td><td style="text-align:left">distance from the city's coordinates to the nearest coastline</td></tr>
<tr><td style="text-align:left">Population</td><td style="text-align:left">Eurostat urb_cpop1; Statistik Austria and INSEE for 8 cities</td><td style="text-align:left">2020 (2013 for Athens)</td><td style="text-align:left">one value per city</td><td style="text-align:left">matched by city name; 12 values filled by hand (section 4)</td></tr>
</table>

Pollution is measured at stations, weather is a model value on a grid, and population is an administrative figure: the three are not equally precise, and nearby cities share the same weather cell. These limits are recalled where they matter.

In [1]:
# Libraries: pandas for tables, numpy for numerical computations.
import pandas as pd
import numpy as np

# Folder with the intermediate tables (the notebook lives in notebooks/, so we go up one level with ..)
INTERIM = "../data/interim/"

# Show slightly wider tables than the default
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

# show() displays tables aligned to the left (defined in src/style.py, shared by all notebooks)
import sys
sys.path.append("../src")
from style import show

## 1. The cities and their means

The file `coverage_cities_2016_2023.csv` has one row per city with: how many valid years it has for each pollutant, and the 2016–2023 means. The column `ok_6years` is `True` for cities with at least 6 valid years out of 8 **for each** of PM10, NO₂ and ozone. The years are counted separately for each pollutant, so they need not be the same years: the next cell counts the years that are valid for all three at once.

In [2]:
cities = pd.read_csv(INTERIM + "coverage_cities_2016_2023.csv")
print("Cities in the file:", len(cities))

# Keep only the cities with enough data
cities = cities[cities["ok_6years"]].copy()
print("Cities with at least 6 valid years out of 8:", len(cities))

show(cities.head())

Cities in the file: 573
Cities with at least 6 valid years out of 8: 289


,country,city,lat,lon,altitude,years_PM10,years_PM25,years_NO2,years_O3,pm10_mean,days_above_50,pm25_mean,no2_mean,o3_p93_max8h,ok_5years,ok_6years,ok_7years
0,AT,Graz,47.07,15.43,377.60,8.00,8.00,8.00,8.00,20.89,13.18,15.10,23.28,114.59,True,True,True
1,AT,Innsbruck,47.27,11.39,597.89,8.00,8.00,8.00,8.00,15.14,3.41,9.54,24.71,111.72,True,True,True
2,AT,Klagenfurt,46.63,14.30,460.00,8.00,8.00,8.00,8.00,16.65,4.62,11.04,19.22,109.97,True,True,True
3,AT,Linz,48.29,14.31,262.15,8.00,8.00,8.00,8.00,18.13,7.31,12.57,23.75,112.99,True,True,True
4,AT,Salzburg,47.81,13.04,423.75,8.00,8.00,8.00,8.00,14.06,4.38,9.18,20.30,117.18,True,True,True


### Cities outside continental Europe

The list includes some French overseas territories (French Guiana, Guadeloupe, Réunion) and two cities in the Canary Islands. They have a tropical or subtropical climate and Saharan dust: comparing them with Milan makes no sense for our question. **Choice:** we remove them (recorded in the README decision log).

In [3]:
outside_europe = ["Cayenne", "Les Abymes", "Saint Denis", "Santa Cruz de Tenerife", "Telde"]
removed = cities[cities["city"].isin(outside_europe)][["country", "city", "lat", "lon"]]

cities = cities[~cities["city"].isin(outside_europe)]
print("Cities removed:", len(removed), "- cities left:", len(cities))
show(removed)

Cities removed: 5 - cities left: 284


,country,city,lat,lon
197,ES,Santa Cruz de Tenerife,28.46,-16.28
203,ES,Telde,28.00,-15.41
234,FR,Cayenne,4.93,-52.32
251,FR,Les Abymes,16.24,-61.54
276,FR,Saint Denis,-20.89,55.46


> ✅ **CHECKPOINT 1 — 284 cities.** This is the **final set of cities** of the project: from here on no city is added or removed. All three blocks (exploration, clustering, classification) work on these 284 cities in 20 countries, 57 of them Italian.

### What a record is: sampling point, station, year, city

The intermediate tables were built in four steps, from the smallest unit to the city:

1. **Sampling point**: one instrument measuring one pollutant (one file in the EEA data). A station can have more than one sampling point for the same pollutant, for example two PM10 instruments working side by side.
2. **Station-year**: a sampling point counts for a year if it has at least 75% of the days (PM) or of the hours (NO₂, ozone). Each row of `coverage_pm_station_years.csv` and `signature_pm_station_years.csv` is one valid sampling point in one year.
3. **City-year**: the mean of the valid rows of the city in that year.
4. **City**: the mean of the city-years 2016–2023.

So a station with two PM10 sampling points enters step 3 twice and weighs double in that year. The next cell counts how often this happens and how much it changes.

In [4]:
# Years valid for all three pollutants at once (the rule above counts them separately)
pm_years = pd.read_csv(INTERIM + "coverage_pm_station_years.csv")
pm_years = pm_years[pm_years["year"].between(2016, 2023) & (pm_years["valid_days"] >= 274)]
gas_years = pd.read_csv(INTERIM + "coverage_no2_o3_station_years.csv")
gas_years = gas_years[gas_years["year"].between(2016, 2023)]

def valid_years(table, pollutant):
    # set of valid years of each city for one pollutant
    return table[table["pollutant"] == pollutant].groupby(["country", "city"])["year"].apply(set)

sets = [valid_years(pm_years, "PM10"), valid_years(gas_years, "NO2"), valid_years(gas_years, "O3")]
keys = list(zip(cities["country"], cities["city"]))
common = pd.Series([len(sets[0][k] & sets[1][k] & sets[2][k]) for k in keys], index=cities["city"].values)
print("Cities by number of years valid for PM10, NO2 and ozone together:")
print(common.value_counts().sort_index().to_string())
print("Cities with fewer than 6 common years:", list(common[common < 6].index))

Cities by number of years valid for PM10, NO2 and ozone together:
5      1
6     25
7     46
8    212
Cities with fewer than 6 common years: ['Ploiesti']


In [5]:
# Stations with more than one PM sampling point in the same year, among the cities kept
pm10_years = pm_years[pm_years["pollutant"] == "PM10"].merge(cities[["country", "city"]])
twice = pm10_years[pm10_years.duplicated(["eoi", "year"], keep=False)]
print("PM10 station-years measured by two sampling points:", twice.drop_duplicates(["eoi", "year"]).shape[0],
      "in", twice["city"].nunique(), "cities")

def city_values(table):
    # step 3 and step 4: mean per city and year, then mean over the years
    per_year = table.groupby(["country", "city", "year"])[["mean", "days_above_50"]].mean()
    return per_year.groupby(["country", "city"]).mean()

as_now = city_values(pm10_years)                                            # every sampling point counts once
by_station = city_values(pm10_years.groupby(["country", "city", "eoi", "year"], as_index=False)
                         [["mean", "days_above_50"]].mean())                # every station counts once
print(f"Largest change in PM10 mean: {(as_now['mean'] - by_station['mean']).abs().max():.2f} µg/m³")
print(f"Largest change in days above 50: {(as_now['days_above_50'] - by_station['days_above_50']).abs().max():.1f} days")
print("Cities whose 'over 35 days' answer would change:",
      ((as_now["days_above_50"] > 35) != (by_station["days_above_50"] > 35)).sum())

PM10 station-years measured by two sampling points: 436 in 53 cities
Largest change in PM10 mean: 0.54 µg/m³
Largest change in days above 50: 1.9 days
Cities whose 'over 35 days' answer would change: 0


**What we see.** Ploiești is the only city with fewer than 6 years valid for all three pollutants together (5); every other city has at least 6. In 53 cities some PM10 station is measured by two sampling points in some years (436 station-years). Counting every station once instead changes the PM10 mean by at most about half a µg/m³ and the days above 50 by less than 2, and no city crosses the 35-day line. **Choice:** we keep the aggregation as it is, and we state it; notebook 03 (section 8) checks that the clustering does not change either.

In [6]:
# Keep and rename only the columns we need, with clear names
cities = cities[["country", "city", "lat", "lon", "altitude",
                 "pm10_mean", "pm25_mean", "no2_mean", "o3_p93_max8h", "days_above_50"]]
cities = cities.rename(columns={
    "o3_p93_max8h": "o3_peaks",                # ozone peaks: 93rd percentile of the daily 8-hour maximum, whole year
    "days_above_50": "pm10_days_above_50",     # days per year with PM10 above 50 µg/m³ (mean over the years)
})
show(cities.describe(), 1)

,lat,lon,altitude,pm10_mean,pm25_mean,no2_mean,o3_peaks,pm10_days_above_50
count,284.0,284.0,284.0,284.0,244.0,284.0,284.0,284.0
mean,47.0,8.7,177.1,19.8,12.5,17.7,113.2,12.4
std,4.8,8.0,185.0,5.7,4.5,5.4,11.9,15.6
min,36.7,-9.3,0.0,8.5,4.3,7.0,81.8,0.1
25%,43.6,3.2,41.2,15.4,9.5,13.7,106.5,2.6
50%,47.5,9.2,112.0,18.3,11.0,17.4,113.2,4.7
75%,50.6,14.2,252.4,23.5,14.6,21.0,118.9,15.3
max,65.0,28.1,1150.0,39.1,26.3,35.1,157.0,73.2


**The ozone column.** `o3_peaks` is, for each year, the **93rd percentile of the daily maximum 8-hour mean, computed over all the days of the year**, then averaged over the years. It is the indicator the EU uses for its ozone target value (120 µg/m³ not to be exceeded on more than 25 days a year: the 26th highest day is about the 93rd percentile). It is an **annual indicator of ozone peaks**, not a summer mean: the peaks happen in the sunny months, but no month is excluded from the computation.

## 2. The pollution "signature"

Besides mean levels, we want variables that tell **what kind** of pollution a city has. They come from two tables with **one row per station and per year**:

- `signature_pm_station_years.csv` (PM10 and PM2.5): yearly mean, winter mean (November–February), summer mean (May–August), weekday mean and weekend mean;
- `signature_no2_station_years.csv` (NO₂): the same columns.

We turn them into **one row per city** in two steps: first the mean of the city's stations in each year, then the mean over the years (so a city with many stations does not weigh more).

In [7]:
pm = pd.read_csv(INTERIM + "signature_pm_station_years.csv")
no2 = pd.read_csv(INTERIM + "signature_no2_station_years.csv")
show(pm.head())

,country,city,pollutant,eoi,year,mean,mean_winter,mean_summer,mean_weekdays,mean_weekend
0,AT,Klagenfurt,PM10,AT2KA71,2016,18.05,27.69,10.82,17.86,18.53
1,AT,Klagenfurt,PM10,AT2KA71,2017,18.33,28.91,11.82,18.85,17.03
2,AT,Klagenfurt,PM10,AT2KA71,2018,18.90,22.95,13.97,19.10,18.41
3,AT,Klagenfurt,PM10,AT2KA71,2019,15.67,19.87,13.15,15.94,14.99
4,AT,Klagenfurt,PM10,AT2KA71,2020,14.16,19.49,9.60,14.11,14.26


In [8]:
columns = ["mean", "mean_winter", "mean_summer", "mean_weekdays", "mean_weekend"]

def per_city(table):
    # step 1: mean of the stations, per city and year
    per_year = table.groupby(["country", "city", "year"])[columns].mean()
    # step 2: mean over the years, per city
    return per_year.groupby(["country", "city"]).mean().reset_index()

pm10_c = per_city(pm[pm["pollutant"] == "PM10"])
pm25_c = per_city(pm[pm["pollutant"] == "PM2.5"])
no2_c = per_city(no2)

# Example: Milan
show(pm10_c[pm10_c["city"] == "Milano (greater city)"], 1)

,country,city,mean,mean_winter,mean_summer,mean_weekdays,mean_weekend
299,IT,Milano (greater city),29.9,43.8,17.7,30.3,28.8


### The four signature variables

<table style="margin-left:0; margin-right:auto">
<tr><th style="text-align:left">Variable</th><th style="text-align:left">How it is computed</th><th style="text-align:left">What it tells</th></tr>
<tr><td style="text-align:left"><code>pm10_winter_summer</code></td><td style="text-align:left">winter PM10 / summer PM10</td><td style="text-align:left">High values = smog that comes in winter: heating and stagnant air</td></tr>
<tr><td style="text-align:left"><code>no2_winter_summer</code></td><td style="text-align:left">winter NO₂ / summer NO₂</td><td style="text-align:left">Same as above, for the traffic pollutant</td></tr>
<tr><td style="text-align:left"><code>no2_weekend_drop</code></td><td style="text-align:left">(weekday NO₂ − weekend NO₂) / weekday NO₂, in %; weekdays = Monday–Friday, weekend = Saturday and Sunday</td><td style="text-align:left">How much weekday traffic weighs</td></tr>
<tr><td style="text-align:left"><code>pm25_pm10_ratio</code></td><td style="text-align:left">mean PM2.5 / mean PM10</td><td style="text-align:left">Fine particles (combustion) versus coarser particles (resuspension, dust)</td></tr>
</table>

In [9]:
signature = pd.DataFrame({"country": pm10_c["country"], "city": pm10_c["city"],
                          "pm10_winter_summer": pm10_c["mean_winter"] / pm10_c["mean_summer"]})

tmp = no2_c.assign(no2_winter_summer=no2_c["mean_winter"] / no2_c["mean_summer"],
                   no2_weekend_drop=100 * (no2_c["mean_weekdays"] - no2_c["mean_weekend"]) / no2_c["mean_weekdays"])
signature = signature.merge(tmp[["country", "city", "no2_winter_summer", "no2_weekend_drop"]], on=["country", "city"], how="outer")

# PM2.5/PM10 ratio: we use the means of the same cities
ratio = pm25_c[["country", "city", "mean"]].merge(pm10_c[["country", "city", "mean"]], on=["country", "city"],
                                                  suffixes=("_pm25", "_pm10"))
ratio["pm25_pm10_ratio"] = ratio["mean_pm25"] / ratio["mean_pm10"]
signature = signature.merge(ratio[["country", "city", "pm25_pm10_ratio"]], on=["country", "city"], how="left")

cities = cities.merge(signature, on=["country", "city"], how="left")
print("Cities:", len(cities), "- columns:", cities.shape[1])
show(cities[cities["city"].isin(["Milano (greater city)", "Kraków", "Paris (greater city)", "Madrid"])][
    ["city", "pm10_winter_summer", "no2_winter_summer", "no2_weekend_drop", "pm25_pm10_ratio"]])

Cities: 284 - columns: 14


,city,pm10_winter_summer,no2_winter_summer,no2_weekend_drop,pm25_pm10_ratio
100,Madrid,0.88,1.92,21.86,0.53
157,Paris (greater city),1.22,1.67,20.97,0.62
205,Milano (greater city),2.48,2.37,16.40,0.73
253,Kraków,2.11,1.34,19.48,0.74


> ✅ **CHECKPOINT 2 — pollution done.** Still 284 cities, now with the mean levels and the four signature variables: everything block B needs for the clustering.

## 3. The weather

**Source:** NASA POWER (MERRA-2 reanalysis, cells of about 50 km), **monthly** means 2016–2023 at the coordinates of each city. We used NASA POWER instead of Open-Meteo because it returns monthly means directly, with a single request per city.

The file is "long": one row per city, year, month and variable. The variables:

<table style="margin-left:0; margin-right:auto">
<tr><th style="text-align:left">Code</th><th style="text-align:left">Meaning</th><th style="text-align:left">Unit</th></tr>
<tr><td style="text-align:left"><code>T2M</code></td><td style="text-align:left">temperature at 2 m</td><td style="text-align:left">°C</td></tr>
<tr><td style="text-align:left"><code>T2M_RANGE</code></td><td style="text-align:left">daily temperature range</td><td style="text-align:left">°C</td></tr>
<tr><td style="text-align:left"><code>PRECTOTCORR</code></td><td style="text-align:left">precipitation</td><td style="text-align:left">mm per day</td></tr>
<tr><td style="text-align:left"><code>WS10M</code></td><td style="text-align:left">wind at 10 m</td><td style="text-align:left">m/s</td></tr>
<tr><td style="text-align:left"><code>RH2M</code></td><td style="text-align:left">relative humidity</td><td style="text-align:left">%</td></tr>
<tr><td style="text-align:left"><code>PS</code></td><td style="text-align:left">surface pressure</td><td style="text-align:left">kPa</td></tr>
<tr><td style="text-align:left"><code>PBLTOP</code></td><td style="text-align:left">pressure at the top of the mixing layer</td><td style="text-align:left">kPa</td></tr>
<tr><td style="text-align:left"><code>ALLSKY_SFC_SW_DWN</code></td><td style="text-align:left">solar radiation at the surface</td><td style="text-align:left">MJ/m² per day</td></tr>
<tr><td style="text-align:left"><code>FROST_DAYS</code></td><td style="text-align:left">frost days</td><td style="text-align:left">days in the month</td></tr>
</table>

In [10]:
weather = pd.read_csv(INTERIM + "weather_cities_monthly.csv")
show(weather.head())

,country,city,year,month,variable,value
0,AT,Graz,2016,1,ALLSKY_SFC_SW_DWN,4.34
1,AT,Graz,2016,2,ALLSKY_SFC_SW_DWN,6.12
2,AT,Graz,2016,3,ALLSKY_SFC_SW_DWN,10.53
3,AT,Graz,2016,4,ALLSKY_SFC_SW_DWN,15.05
4,AT,Graz,2016,5,ALLSKY_SFC_SW_DWN,16.82


### The mixing layer

It is the layer of air near the ground in which pollutants mix. When it is **thin** (cold winters, stagnant air, temperature inversion), smog stays trapped near the ground. NASA POWER does not give its depth in metres but the **pressure** at its top: the depth comes from the difference between the surface pressure and the pressure at the top. Near the ground 1 hPa corresponds to about 8.5 metres of air, so:

depth (m) ≈ (PS − PBLTOP) × 10 × 8.5  (×10 because the values are in kPa)

In [11]:
# From "long" to "wide" format: one column per variable
wide = weather.pivot_table(index=["country", "city", "year", "month"], columns="variable", values="value").reset_index()
wide["mixing_layer_m"] = (wide["PS"] - wide["PBLTOP"]) * 10 * 8.5

# Means over the whole year and over the winter months only (November-February), 2016-2023
winter = wide[wide["month"].isin([11, 12, 1, 2])]
year_mean = wide.groupby(["country", "city"]).mean(numeric_only=True)
win = winter.groupby(["country", "city"]).mean(numeric_only=True)

weather_c = pd.DataFrame({
    "temperature": year_mean["T2M"],
    "temperature_winter": win["T2M"],
    "diurnal_range": year_mean["T2M_RANGE"],
    "rain_mm_day": year_mean["PRECTOTCORR"],
    "wind": year_mean["WS10M"],
    "wind_winter": win["WS10M"],
    "humidity": year_mean["RH2M"],
    "solar_radiation": year_mean["ALLSKY_SFC_SW_DWN"],
    "frost_days_year": year_mean["FROST_DAYS"] * 12,
    "mixing_layer_m": year_mean["mixing_layer_m"],
    "mixing_layer_winter_m": win["mixing_layer_m"],
}).reset_index()

cities = cities.merge(weather_c, on=["country", "city"], how="left")
print("Cities:", len(cities), "- columns:", cities.shape[1])
show(cities[cities["city"].isin(["Milano (greater city)", "Kraków", "Paris (greater city)", "Madrid"])][
    ["city", "temperature_winter", "wind_winter", "mixing_layer_winter_m", "rain_mm_day"]], 1)

Cities: 284 - columns: 25


,city,temperature_winter,wind_winter,mixing_layer_winter_m,rain_mm_day
100,Madrid,5.9,3.6,377.6,1.4
157,Paris (greater city),5.0,5.0,429.5,1.9
205,Milano (greater city),5.6,2.1,448.1,2.6
253,Kraków,0.4,4.5,393.8,2.2


> ✅ **CHECKPOINT 3 — weather done.** Still 284 cities, with 11 weather variables (yearly and winter means 2016–2023).

## 4. Geography and population

`geography_cities.csv` contains:
- `dist_sea_km`: distance from the nearest coastline, computed with the Natural Earth coastline (1:50 million scale);
- `population`: inhabitants from the Eurostat city statistics (table urb_cpop1, year 2020 when available).

In [12]:
geo = pd.read_csv(INTERIM + "geography_cities.csv")
cities = cities.merge(geo[["country", "city", "dist_sea_km", "population"]], on=["country", "city"], how="left")
print("Cities without population after the Eurostat matching:", cities["population"].isna().sum())
show(cities[["city", "dist_sea_km", "population"]].sort_values("dist_sea_km", ascending=False).head(8), 1)

Cities without population after the Eurostat matching: 12


,city,dist_sea_km,population
177,Debrecen,585.5,201112.0
261,Przemysl,560.9,58625.0
264,Rzeszów,516.8,194374.0
278,Baia Mare,510.5,145444.0
266,Tarnów,497.4,106596.0
22,Ostrava,494.3,287968.0
25,Zlín,489.9,74935.0
253,Kraków,483.8,800531.0


### Filling the 12 missing populations

For 12 cities the matching with Eurostat found no value. We filled them **only from official sources**, listed with year and source in `data/external/population_missing_official.csv`:
- **Geneva, Annemasse, Cherbourg:** Eurostat does have them, under slightly different names (e.g. "Geneva" instead of "Genève"), so the matching missed them. Value for 2020.
- **The 6 Austrian cities:** Eurostat stops in 2014 for Austria. We use **Statistik Austria**, population on 1 January 2020 (Austrian cities are municipalities, as in Eurostat).
- **Saint-Nazaire, Saint-Quentin:** not Eurostat cities. We use **INSEE**, legal municipal population 2020.
- **Athens:** Eurostat has the "greater city" only until 2013; we keep the 2013 value so the definition stays the same as for the other greater cities.

Population is only a rough measure of city size (each country defines "city" differently), and it is used on a logarithmic scale, so these small differences of year do not matter.

In [13]:
official = pd.read_csv("../data/external/population_missing_official.csv")
cities = cities.merge(official[["country", "city", "population"]].rename(columns={"population": "population_official"}),
                      on=["country", "city"], how="left")
cities["population"] = cities["population"].fillna(cities["population_official"])
cities = cities.drop(columns="population_official")
print("Cities without population now:", cities["population"].isna().sum())
show(official[["country", "city", "population", "year", "source"]])

Cities without population now: 0


,country,city,population,year,source
0,AT,Graz,291072,2020,Statistik Austria - Bevoelkerung zu Jahresbeginn 2020 (OGD_bevstandjbab2002_BevStand_2020)
1,AT,Innsbruck,131961,2020,Statistik Austria - Bevoelkerung zu Jahresbeginn 2020 (OGD_bevstandjbab2002_BevStand_2020)
2,AT,Klagenfurt,101300,2020,Statistik Austria - Bevoelkerung zu Jahresbeginn 2020 (OGD_bevstandjbab2002_BevStand_2020)
3,AT,Linz,206595,2020,Statistik Austria - Bevoelkerung zu Jahresbeginn 2020 (OGD_bevstandjbab2002_BevStand_2020)
4,AT,Salzburg,155021,2020,Statistik Austria - Bevoelkerung zu Jahresbeginn 2020 (OGD_bevstandjbab2002_BevStand_2020)
5,AT,Wien,1911191,2020,Statistik Austria - Bevoelkerung zu Jahresbeginn 2020 (OGD_bevstandjbab2002_BevStand_2020)
6,CH,Genève (greater city),415360,2020,Eurostat urb_cpop1 (city CH003C Geneva (greater city))
7,FR,Annemasse,62880,2020,Eurostat urb_cpop1 (city FR076C Annemasse-Geneva FR part (greater city))
8,FR,Cherbourg,77789,2020,Eurostat urb_cpop1 (city FR056C Cherbourg-en-Cotentin)
9,FR,Saint-Nazaire,71887,2020,INSEE - Populations legales 2020 (population municipale; commune 44184)


> ✅ **CHECKPOINT 4 — geography done.** Still 284 cities; distance from the sea and population are complete for all of them.

## 5. The target for block C

**Where the limit comes from.** The EU air quality directive in force, **Directive 2008/50/EC** (Annex XI), sets two limit values for PM10 to protect human health:
- a **daily** limit: the daily mean must not exceed **50 µg/m³ more than 35 times in a calendar year**;
- an **annual** limit: the yearly mean must not exceed **40 µg/m³**.

The daily limit is the one that matters in practice: in our data no city has a yearly mean above 40 µg/m³ (the highest is Rybnik, 39.1), while 30 cities have more than 35 days above 50 µg/m³ on our indicator (below). "35 days" means that 50 µg/m³ may be exceeded on fewer than 10% of the days of the year (35 of 365). The new **Directive (EU) 2024/2881** lowers these values from 2030: 45 µg/m³ on no more than 18 days, and 20 µg/m³ as yearly mean, closer to the WHO 2021 guidelines.

**Our column.** `pm10_days_above_50` is, for each city, the mean over 2016–2023 of the yearly number of days above 50 µg/m³ at its background stations. We create a yes/no column: does the city, on average, exceed **35 days**? It is an indicator built for our comparison, not a legal assessment: legally the limit is checked station by station, year by year, and roadside stations count too. In figures and slides we therefore call it **"PM10 exceedance indicator (background stations, mean 2016–2023)"**, not "breaks the EU limit".

In [14]:
cities["exceeds_pm10_limit"] = cities["pm10_days_above_50"] > 35

italy = cities["country"] == "IT"
over = cities["exceeds_pm10_limit"]
print("Cities over the PM10 exceedance indicator:", over.sum(), "of", len(cities))
print("  Italy:          ", (over & italy).sum(), "of", italy.sum(), f"({(over & italy).sum() / italy.sum():.0%})")
print("  Rest of Europe: ", (over & ~italy).sum(), "of", (~italy).sum(), f"({(over & ~italy).sum() / (~italy).sum():.0%})")
print("Highest yearly mean PM10:", cities["pm10_mean"].max(), "µg/m³ (annual limit 40)")

Cities over the PM10 exceedance indicator: 30 of 284
  Italy:           19 of 57 (33%)
  Rest of Europe:  11 of 227 (5%)
Highest yearly mean PM10: 39.07 µg/m³ (annual limit 40)


> 📌 **19 of the 57 Italian cities (one in three) are over the PM10 exceedance indicator, against 11 of the 227 cities in the rest of Europe (5%).** Italy has 20% of the cities in our sample but almost two thirds of those over the indicator, and most of them are in the Po Valley. This is the imbalance that block C tries to explain with weather and geography.

> ✅ **CHECKPOINT 5 — target done.** 30 cities over the indicator (True), 254 not (False).

## 6. Checks

Before saving we look at how many values are missing per column, and whether there are impossible values (negative numbers, absurd ratios).

In [15]:
missing = cities.isna().sum()
# Keep only the columns with at least one missing value, and give the count column a name
show(missing[missing > 0].to_frame("cities with missing value"))

,cities with missing value
pm25_mean,40
pm25_pm10_ratio,40


**Why PM2.5 is missing in 40 cities** (checked station by station): 32 cities have no background station measuring PM2.5 in 2016–2023; the other 8 (Köln, Lorca, Gera, Palma de Mallorca, Pamplona, Logroño, Santander, Debrecen) have one, but it measures at most 60–183 days a year, probably one sample every 2–3 days, below our rule of 274 valid days. Rybnik and Zabrze, the two cities with the highest PM10, are among the 40: block B should drop the two PM2.5 columns rather than these cities.

In [16]:
# Summary of all numerical columns: minimum, maximum, mean...
show(cities.describe().T)

,count,mean,std,min,25%,50%,75%,max
lat,284.00,47.02,4.77,36.69,43.62,47.51,50.55,65.03
lon,284.00,8.74,7.95,-9.35,3.23,9.21,14.24,28.05
altitude,284.00,177.12,185.01,0.00,41.21,112.00,252.38,1150.00
pm10_mean,284.00,19.77,5.67,8.47,15.41,18.30,23.50,39.07
pm25_mean,244.00,12.54,4.46,4.28,9.53,11.04,14.58,26.26
no2_mean,284.00,17.74,5.37,6.96,13.73,17.41,21.04,35.10
o3_peaks,284.00,113.20,11.87,81.82,106.50,113.19,118.86,157.03
pm10_days_above_50,284.00,12.38,15.57,0.06,2.59,4.73,15.34,73.19
pm10_winter_summer,284.00,1.35,0.44,0.59,1.07,1.24,1.53,2.98
no2_winter_summer,284.00,1.88,0.47,0.95,1.53,1.78,2.12,3.63


### A first check on the weather

Do the weather variables make sense? We look at the correlation between some of them and PM10. We expect **more wind** to mean **less PM10** (negative correlation).

In [17]:
weather_cols = ["wind_winter", "temperature_winter", "rain_mm_day", "mixing_layer_winter_m", "altitude", "dist_sea_km"]
show(cities[weather_cols].corrwith(cities["pm10_mean"]).to_frame("correlation with PM10"))

,correlation with PM10
wind_winter,-0.23
temperature_winter,0.10
rain_mm_day,-0.24
mixing_layer_winter_m,0.08
altitude,-0.13
dist_sea_km,-0.02


**Note.** Winter wind goes in the expected direction. The mixing-layer depth, instead, is almost uncorrelated with PM10: it is a monthly mean over cells of about 50 km that mixes day and night, so it misses the night-time inversions of the Po Valley. Notebook 02 (section 5.3) shows that the yearly value follows the climate rather than stagnant air, so it is **not used** in the models; the winter value enters only the random forest of block C, as a weak variable. **Winter wind** is our measure of stagnant air.

## 7. Saving

The final table goes to `data/processed/cities.csv`. Notebooks 02 (exploration), 03 (clustering) and 04 (classification) start from here.

**Columns available to each block** (the final choices, and why, are explained in notebooks 03 and 04):
- **block B, clustering (signature):** `pm10_mean`, `pm25_mean`, `no2_mean`, `o3_peaks`, `pm10_days_above_50`, `pm10_winter_summer`, `no2_winter_summer`, `no2_weekend_drop`, `pm25_pm10_ratio`;
- **block C, classification (weather and geography):** `temperature`, `temperature_winter`, `diurnal_range`, `rain_mm_day`, `wind`, `wind_winter`, `humidity`, `solar_radiation`, `frost_days_year`, `mixing_layer_m`, `mixing_layer_winter_m`, `altitude`, `lat`, `lon`, `dist_sea_km`, `population`; target: `exceeds_pm10_limit`;
- **block A, exploration:** all of them.

In [18]:
cities = cities.round(3).sort_values(["country", "city"])
cities.to_csv("../data/processed/cities.csv", index=False)
print("Saved: data/processed/cities.csv —", cities.shape[0], "cities,", cities.shape[1], "columns")

Saved: data/processed/cities.csv — 284 cities, 28 columns


> ✅ **CHECKPOINT 6 — final table saved.** `data/processed/cities.csv`: **284 cities × 28 columns**. Only missing values: PM2.5 and PM2.5/PM10 in 40 cities (see section 6). This is the starting point of notebooks 02, 03 and 04.